# CRCBAC – Baseline de Execução no Google Colab

Este notebook executa o baseline **CRCBAC** (Context-aware Role-Capability-Based Access Control) completamente no Colab, usando **MongoDB Atlas** como banco de dados.

**Etapas:**
1. Instalar dependências
2. Conectar ao MongoDB Atlas
3. Clonar o repositório
4. Corrigir URIs do MongoDB nos scripts
5. Fazer seed do banco
6. Rodar o gateway CoAP em background
7. Smoke test (1 request)
8. Benchmark (50 requests → CSV)
9. Visualizar resultados

## Célula 1 – Instalar pymongo

In [1]:
!pip install pymongo[srv] -q

## Célula 2 – Conectar ao MongoDB Atlas

In [2]:
from pymongo import MongoClient

# ⚠️  Cole aqui a sua URI do MongoDB Atlas (não commitar com credenciais reais)
# Formato: mongodb+srv://<user>:<password>@<cluster>.mongodb.net/?appName=<app>
MONGO_URI = "mongodb+srv://<USER>:<PASSWORD>@<CLUSTER>.mongodb.net/?appName=<APP>"

client = MongoClient(MONGO_URI)

print("Connected successfully.")print("Databases:", client.list_database_names())

Connected successfully.
Databases: ['CRBAC_Policies', 'admin', 'local']


## Célula 3 – Clonar o repo e instalar dependências

In [3]:
!git clone https://github.com/Taimisson/crcbaccode.git
%cd crcbaccode
!pip install -q -r CRCBAC_runner_kit/requirements.txt

c:\Users\taimi\Desktop\CRCBAC_code\crcbaccode


Cloning into 'crcbaccode'...
c:\Users\taimi\Desktop\CRCBAC_code\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## Célula 4 – Verificar onde estão as referências ao MongoDB local

In [5]:
!grep -R "MongoClient(" .
!grep -R "mongodb://localhost" .
!grep -R "27017" .

'grep' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.
'grep' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.
'grep' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


## Célula 5 – Exportar a URI do Atlas como variável de ambiente

In [ ]:
import os
# ⚠️  Cole aqui a sua URI do MongoDB Atlas (não commitar com credenciais reais)
os.environ["MONGO_URI"] = "mongodb+srv://<USER>:<PASSWORD>@<CLUSTER>.mongodb.net/?appName=<APP>"
print("MONGO_URI set:", os.environ["MONGO_URI"][:40], "...")

## Célula 6 – Patch seed_mongo.py (adicionar `import os` e trocar URI)

In [ ]:
# 1) Adicionar import os após 'from __future__ import annotations'
!grep -q "^import os" CRCBAC_runner_kit/seed_mongo.py \
  || sed -i '/from __future__ import annotations/a import os' CRCBAC_runner_kit/seed_mongo.py

# 2) Trocar o URI hardcoded pela variável de ambiente
!sed -i 's|MONGO_URI = "mongodb://localhost:27017/"|MONGO_URI = os.environ["MONGO_URI"]|g' \
  CRCBAC_runner_kit/seed_mongo.py

print("Patch aplicado. Verificando linhas 1-25:")
!sed -n '1,25p' CRCBAC_runner_kit/seed_mongo.py

## Célula 7 – Patch CRBAC_Gateway_Grant_transfer.py

In [ ]:
GATEWAY = "CRBAC_GRT_dataset_code_result/CODE/CRBAC_Gateway_Grant_transfer.py"

# Trocar MongoClient localhost pelo Atlas
!sed -i 's|MongoClient("mongodb://localhost:27017/")|MongoClient(os.environ["MONGO_URI"])|g' {GATEWAY}

# Adicionar import os no início se não existir
!grep -q "^import os" {GATEWAY} \
  || sed -i '1i import os' {GATEWAY}

print("Patch aplicado. Verificando primeiras 20 linhas:")
!sed -n '1,20p' {GATEWAY}

## Célula 8 – Patch conflict_servercode.py

In [ ]:
CONFLICT = "conflict_resolve_code_dataset_result/conflict_servercode.py"

!sed -i 's|MongoClient("mongodb://localhost:27017/")|MongoClient(os.environ["MONGO_URI"])|g' {CONFLICT}
!grep -q "^import os" {CONFLICT} \
  || sed -i '/^import asyncio/a import os' {CONFLICT}

print("Patch conflict_servercode.py OK")

## Célula 9 – Patch Crbac_parallel_gateway_cp1.py

In [ ]:
THROUGHPUT = "Throughput_compared_edm_cdm_code_dataset_result/Crbac_parallel_gateway_cp1.py"

!sed -i 's|MongoClient("mongodb://localhost:27017/")|MongoClient(os.environ["MONGO_URI"])|g' {THROUGHPUT}
!grep -q "^import os" {THROUGHPUT} \
  || sed -i '/^import aiocoap/a import os' {THROUGHPUT}

print("Patch Crbac_parallel_gateway_cp1.py OK")

## Célula 10 – Seed do MongoDB Atlas

In [ ]:
!python CRCBAC_runner_kit/seed_mongo.py

## Célula 11 – Verificar porta UDP 5683 disponível

In [ ]:
import socket
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
try:
    sock.bind(('127.0.0.1', 5683))
    print("✅ Porta 5683 disponível")
    sock.close()
except OSError as e:
    print("❌ Porta 5683 ocupada:", e)
    print("→ Reinicie o runtime do Colab e execute novamente.")

## Célula 12 – Iniciar o gateway CoAP em background

In [ ]:
import subprocess, time, os

env = os.environ.copy()

gateway_process = subprocess.Popen(
    ['python', 'CRBAC_GRT_dataset_code_result/CODE/CRBAC_Gateway_Grant_transfer.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env
)

time.sleep(4)  # Aguarda o servidor iniciar

# Verificar se está rodando
if gateway_process.poll() is None:
    print("✅ Gateway CoAP iniciado (PID:", gateway_process.pid, ")")
else:
    out, _ = gateway_process.communicate()
    print("❌ Gateway falhou ao iniciar. Saída:")
    print(out)

## Célula 13 – Smoke test (1 request Grant → deve retornar 'allow')

In [ ]:
!python send_test_coap.py

**Saída esperada:**
```
=== Response (decoded) ===
Role_n: R6, Role_t: R1
Role hierarchy satisfied
Capability matched,permission:allow
T2: HH:MM:SS
```

## Célula 14 – Benchmark (50 requests → gera CSV)

In [ ]:
!python bench_coap.py

## Célula 15 – Visualizar resultados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('bench_processing_time_ms.csv')
df['processing_time_ms'] = df['processing_time_ms'].astype(float)

print("=" * 40)
print("Estatísticas (processing_time_ms):")
print("=" * 40)
print(df['processing_time_ms'].describe().round(3))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(df['i'], df['processing_time_ms'], marker='o', markersize=3, linewidth=1)
ax1.axhline(df['processing_time_ms'].mean(), color='red', linestyle='--', label=f"Mean: {df['processing_time_ms'].mean():.2f} ms")
ax1.set_xlabel('Sample')
ax1.set_ylabel('Time (ms)')
ax1.set_title('CRCBAC – Decision Latency per Request')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.hist(df['processing_time_ms'], bins=15, edgecolor='black', color='steelblue')
ax2.axvline(df['processing_time_ms'].mean(), color='red', linestyle='--', label=f"Mean: {df['processing_time_ms'].mean():.2f} ms")
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Frequency')
ax2.set_title('CRCBAC – Latency Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('crcbac_benchmark.png', dpi=150)
plt.show()
print("Gráfico salvo em crcbac_benchmark.png")

## Célula 16 – Parar o gateway

In [ ]:
gateway_process.terminate()
gateway_process.wait()
print("✅ Gateway encerrado.")